# Engage Corpus Evaluation - v4 Data (Multi-Label)

This notebook evaluates predictions for v4 data using our proposed multi-label methodology.

**Methodology**: Each user can have MULTIPLE correct subreddits
- Ground truth: Array of shape (n_users, n_items) with binary relevance (1=interacted, 0=not)
- Predictions: Array of shape (n_users, n_items) with prediction scores

**Metrics**:
- **Hit Rate @ 10 (HR@10)**: Rate at which ANY relevant subreddit appears in top 10 recommendations
- **NDCG @ 10**: Normalized Discounted Cumulative Gain at 10 with multi-label relevance

## 1. Setup and Imports

In [ ]:
import numpy as np
import pickle
from pathlib import Path
from typing import Dict, Tuple
import pandas as pd

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configuration

In [ ]:
# Base directory where predictions are stored
BASE_DIR = Path('/content/drive/MyDrive/CIS 5300/predictions')

# Ground truth file
GROUND_TRUTH_FILE = BASE_DIR / 'ground_truth_v4_test.npy'

# Prediction files for v4 data (179936 users, 2000 items)
PREDICTION_FILES = {
    'Cosine Similarity (v4, 128-dim)': BASE_DIR / 'cosine_v4_128.npy',
    'Cosine Similarity (v4, 4096-dim)': BASE_DIR / 'cosine_v4_4096.npy',
    'NCF Model B (v4)': BASE_DIR / 'predictions_model_B.npy',
    'NCF Model C (v4)': BASE_DIR / 'predictions_model_C.npy',
    'NCF Model E (v4 Gated)': BASE_DIR / 'predictions_model_E.npy', # New
}

# Top-K parameter
K = 10

## 4. Evaluation Functions (Multi-Label)

In [ ]:
def hit_rate_at_k_multilabel(predictions: np.ndarray, ground_truth: np.ndarray, k: int = 10) -> float:
    """
    Calculate Hit Rate @ K for multi-label classification.

    HR@K is the rate at which at least one relevant item appears in the top K recommendations.

    Args:
        predictions: Array of shape (n_users, n_items) with prediction scores
        ground_truth: Array of shape (n_users, n_items) with binary relevance (1=interacted, 0=not)
        k: Number of top recommendations to consider

    Returns:
        Hit rate as a float between 0 and 1
    """
    n_users = predictions.shape[0]
    hits = 0

    for i in range(n_users):
        # Get top k predictions for this user
        top_k_items = np.argsort(predictions[i])[::-1][:k]

        # Check if any ground truth items are in top k
        relevant_in_top_k = ground_truth[i, top_k_items].sum()
        if relevant_in_top_k > 0:
            hits += 1

    return hits / n_users


def ndcg_at_k_multilabel(predictions: np.ndarray, ground_truth: np.ndarray, k: int = 10) -> float:
    """
    Calculate Normalized Discounted Cumulative Gain @ K for multi-label classification.

    NDCG@K accounts for multiple relevant items and their positions in the ranking.

    DCG@K = sum_{i=1}^{K} (rel_i / log2(i + 1))
    IDCG@K = DCG@K for perfect ranking (all relevant items at top)
    NDCG@K = DCG@K / IDCG@K

    Args:
        predictions: Array of shape (n_users, n_items) with prediction scores
        ground_truth: Array of shape (n_users, n_items) with binary relevance (1=interacted, 0=not)
        k: Number of top recommendations to consider

    Returns:
        NDCG score as a float between 0 and 1
    """
    n_users = predictions.shape[0]
    ndcg_sum = 0.0

    for i in range(n_users):
        # Get top k predictions for this user
        top_k_items = np.argsort(predictions[i])[::-1][:k]

        # Get relevance scores for top k items
        relevance_scores = ground_truth[i, top_k_items]

        # Calculate DCG
        dcg = 0.0
        for rank, rel in enumerate(relevance_scores, start=1):
            if rel > 0:
                dcg += rel / np.log2(rank + 1)

        # Calculate IDCG (ideal DCG - all relevant items at top)
        num_relevant = int(ground_truth[i].sum())
        if num_relevant > 0:
            # Ideal ranking has all relevant items first
            ideal_relevance = np.array([1] * min(num_relevant, k) + [0] * max(0, k - num_relevant))
            idcg = 0.0
            for rank, rel in enumerate(ideal_relevance, start=1):
                if rel > 0:
                    idcg += rel / np.log2(rank + 1)

            # Calculate NDCG
            if idcg > 0:
                ndcg = dcg / idcg
                ndcg_sum += ndcg

    return ndcg_sum / n_users


def evaluate_predictions(predictions: np.ndarray, ground_truth: np.ndarray, k: int = 10) -> Dict[str, float]:
    """
    Evaluate predictions using both HR@K and NDCG@K for multi-label setting.

    Args:
        predictions: Array of shape (n_users, n_items) with prediction scores
        ground_truth: Array of shape (n_users, n_items) with binary relevance
        k: Number of top recommendations to consider

    Returns:
        Dictionary with 'hr@k' and 'ndcg@k' metrics
    """
    hr = hit_rate_at_k_multilabel(predictions, ground_truth, k)
    ndcg = ndcg_at_k_multilabel(predictions, ground_truth, k)

    return {
        f'hr@{k}': hr,
        f'ndcg@{k}': ndcg
    }

## 5. Load Ground Truth

In [ ]:
print(f"Loading ground truth from: {GROUND_TRUTH_FILE}")
print("Note: This is a large file (1.4 GB) and may take a moment...\n")

ground_truth = np.load(GROUND_TRUTH_FILE)

print(f"Ground truth shape: {ground_truth.shape}")
print(f"Ground truth dtype: {ground_truth.dtype}")
print(f"Number of users: {ground_truth.shape[0]}")
print(f"Number of items: {ground_truth.shape[1]}")

# Analyze ground truth
total_interactions = ground_truth.sum()
avg_interactions_per_user = ground_truth.sum(axis=1).mean()
users_with_interactions = (ground_truth.sum(axis=1) > 0).sum()

print(f"\nGround Truth Statistics:")
print(f"  Total interactions: {int(total_interactions):,}")
print(f"  Average interactions per user: {avg_interactions_per_user:.2f}")
print(f"  Users with at least 1 interaction: {users_with_interactions:,} ({users_with_interactions/ground_truth.shape[0]*100:.2f}%)")
print(f"  Sparsity: {(1 - total_interactions / ground_truth.size) * 100:.2f}%")

## 6. Evaluate All Models

In [ ]:
results = {}

print("=" * 80)
print("ENGAGE CORPUS EVALUATION - v4 DATA (MULTI-LABEL)")
print("=" * 80)
print(f"\nEvaluating metrics: HR@{K} and NDCG@{K}")
print(f"Number of users: {ground_truth.shape[0]:,}")
print(f"Number of items: {ground_truth.shape[1]:,}\n")

for model_name, pred_file in PREDICTION_FILES.items():
    print(f"\n{'='*80}")
    print(f"Model: {model_name}")
    print(f"File: {pred_file.name}")
    print("-" * 80)

    try:
        # Load predictions
        print("Loading predictions (this may take a moment for large files)...")
        predictions = np.load(pred_file)
        print(f"✓ Loaded successfully")
        print(f"Predictions shape: {predictions.shape}")
        print(f"Predictions dtype: {predictions.dtype}")

        # Validate shapes
        if predictions.shape != ground_truth.shape:
            print(f"  WARNING: Shape mismatch!")
            print(f"   Predictions: {predictions.shape}")
            print(f"   Ground truth: {ground_truth.shape}")
            continue

        # Evaluate
        print(f"\nEvaluating (processing {ground_truth.shape[0]:,} users)...")
        model_results = evaluate_predictions(predictions, ground_truth, k=K)
        results[model_name] = model_results

        # Print results
        print(f"\n✓ Results:")
        print(f"  Hit Rate @ {K}:  {model_results[f'hr@{K}']:.4f}  ({model_results[f'hr@{K}']*100:.2f}%)")
        print(f"  NDCG @ {K}:      {model_results[f'ndcg@{K}']:.4f}")

    except FileNotFoundError:
        print(f" ERROR: File not found: {pred_file}")
    except Exception as e:
        print(f" ERROR: {str(e)}")
        import traceback
        traceback.print_exc()

print(f"\n{'='*80}")

## 7. Summary Table

In [ ]:
if results:
    # Create summary DataFrame
    summary_data = []
    for model_name, metrics in results.items():
        summary_data.append({
            'Model': model_name,
            f'HR@{K}': f"{metrics[f'hr@{K}']:.4f} ({metrics[f'hr@{K}']*100:.2f}%)",
            f'NDCG@{K}': f"{metrics[f'ndcg@{K}']:.4f}"
        })

    summary_df = pd.DataFrame(summary_data)

    print("\n" + "="*80)
    print("SUMMARY - v4 DATA (MULTI-LABEL)")
    print("="*80 + "\n")
    print(summary_df.to_string(index=False))
    print("\n" + "="*80)

    # Also display as formatted table
    display(summary_df)
else:
    print("\n  No results to display")

## 8. Best Model Analysis

In [ ]:
if results:
    # Find best model by HR@K
    best_hr_model = max(results.items(), key=lambda x: x[1][f'hr@{K}'])
    best_ndcg_model = max(results.items(), key=lambda x: x[1][f'ndcg@{K}'])

    print("\n BEST MODELS:\n")
    print(f"Best HR@{K}: {best_hr_model[0]}")
    print(f"  Score: {best_hr_model[1][f'hr@{K}']:.4f} ({best_hr_model[1][f'hr@{K}']*100:.2f}%)\n")

    print(f"Best NDCG@{K}: {best_ndcg_model[0]}")
    print(f"  Score: {best_ndcg_model[1][f'ndcg@{K}']:.4f}")

## 9. Detailed Analysis (Optional)

In [ ]:
# Analyze a specific model in detail
# Uncomment and modify to analyze a specific model
# WARNING: This will take longer for v4 data due to the larger dataset size

# MODEL_TO_ANALYZE = 'NCF Model B (v4)'
# predictions = np.load(PREDICTION_FILES[MODEL_TO_ANALYZE])

# # Compute per-user metrics
# user_hits = []
# user_ndcg = []
# user_num_relevant = []
# user_num_hits = []

# print(f"Analyzing {ground_truth.shape[0]:,} users...")
# for i in range(ground_truth.shape[0]):
#     if i % 10000 == 0 and i > 0:
#         print(f"  Processed {i:,} users...")
#
#     top_k_items = np.argsort(predictions[i])[::-1][:K]
#     relevance_scores = ground_truth[i, top_k_items]
#     num_relevant = int(ground_truth[i].sum())
#     num_hits = int(relevance_scores.sum())
#
#     # Check hit
#     hit = 1 if num_hits > 0 else 0
#     user_hits.append(hit)
#     user_num_relevant.append(num_relevant)
#     user_num_hits.append(num_hits)
#
#     # Compute NDCG
#     if num_relevant > 0:
#         dcg = sum(rel / np.log2(rank + 1) for rank, rel in enumerate(relevance_scores, start=1) if rel > 0)
#         ideal_relevance = [1] * min(num_relevant, K) + [0] * max(0, K - num_relevant)
#         idcg = sum(rel / np.log2(rank + 1) for rank, rel in enumerate(ideal_relevance, start=1) if rel > 0)
#         ndcg = dcg / idcg if idcg > 0 else 0
#         user_ndcg.append(ndcg)
#     else:
#         user_ndcg.append(0.0)

# print(f"\nDetailed analysis for: {MODEL_TO_ANALYZE}")
# print(f"Total users: {len(ground_truth):,}")
# print(f"Users with hits: {sum(user_hits):,} ({sum(user_hits)/len(user_hits)*100:.2f}%)")
# print(f"Users with no hits: {len(user_hits) - sum(user_hits):,} ({(len(user_hits) - sum(user_hits))/len(user_hits)*100:.2f}%)")
# print(f"\nAverage relevant items per user: {np.mean(user_num_relevant):.2f}")
# print(f"Average hits per user: {np.mean(user_num_hits):.2f}")
# print(f"Average NDCG: {np.mean(user_ndcg):.4f}")
# print(f"Median NDCG: {np.median(user_ndcg):.4f}")
# print(f"NDCG std dev: {np.std(user_ndcg):.4f}")

## 10. Compare Top-K Values (Optional)

In [ ]:
# Compare performance at different K values
# Uncomment to run this analysis

# MODEL_TO_ANALYZE = 'NCF Model B (v4)'
# K_VALUES = [1, 3, 5, 10, 20]

# predictions = np.load(PREDICTION_FILES[MODEL_TO_ANALYZE])

# print(f"Analyzing {MODEL_TO_ANALYZE} at different K values...\n")

# k_results = []
# for k in K_VALUES:
#     print(f"Evaluating at K={k}...")
#     metrics = evaluate_predictions(predictions, ground_truth, k=k)
#     k_results.append({
#         'K': k,
#         'HR': metrics[f'hr@{k}'],
#         'NDCG': metrics[f'ndcg@{k}']
#     })

# k_df = pd.DataFrame(k_results)
# print("\n" + "="*60)
# print(f"Performance at Different K Values - {MODEL_TO_ANALYZE}")
# print("="*60)
# display(k_df)